# Дообучение RuBERT-tiny2 для классификации спам-сообщений

Ноутбук выполняет дообучение модели `cointegrated/rubert-tiny2` (29M параметров) на собранном датасете.

Особенности обучения:
- **FP16** — смешанная точность для ускорения обучения на GPU
- **FocalLoss** — функция потерь, фокусирующаяся на сложных примерах
- **TemperatureScaler** — калибровка вероятностей после обучения
- **EarlyStopping** — ранняя остановка при отсутствии улучшения F1

Обучение выполняется на трёх вариантах текста:
- исходный текст (raw)
- нормализованный текст (normalized)
- предобработанный текст (preprocessed)

Фильтрация: остаются только сообщения, содержащие преимущественно кириллицу.

## Импорт необходимых библиотек

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    pipeline,
)
from datasets import Dataset

try:
    _project_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
except NameError:
    _cwd = os.getcwd()
    _project_root = _cwd
    while _project_root != '/' and not os.path.isdir(os.path.join(_project_root, 'src')):
        _project_root = os.path.dirname(_project_root)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from src.models.transformer import (
    FocalLossTrainer,
    TemperatureScaler,
    tokenize_function,
    compute_metrics,
    is_mostly_cyrillic,
    get_training_args,
    get_model_config,
    prepare_text_variants,
    benchmark_cpu_inference,
)
from src.config import PROCESSED_DIR, MODELS_DIR


## Чтение обработанного датасета

Загружается `preprocessed.csv` из `data/processed/`.

In [2]:
df = pd.read_csv(PROCESSED_DIR / 'preprocessed.csv', index_col=0)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 77523 entries, Добрый день! Отличается ли перечень необходимых документов для иностранных граждан? to Прикольно, что даже такие важные моменты как прием в учебные заведения можно связывать со знаками зодиака. Это напоминает мне о том, как натальная карта может раскрыть сильные стороны наших детей и их подход к учебе. Очень полезно учитывать, какими качествами обладают наши малышата в зависимости от их знака. Вот бы использовать для этого @Astrologyupgobot, он точно поможет разгадать гороскоп и понять, на что акцентировать внимание!
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   label              77523 non-null  int64  
 1   _source            77523 non-null  object 
 2   emojis             77523 non-null  int64  
 3   newlines           77523 non-null  int64  
 4   whitespaces        77523 non-null  int64  
 5   links              77523 non-null  int64  


## Подготовка вариантов текста

Из датасета выделяются три варианта текста для дообучения:
- **raw** — исходный текст (колонка `text_raw`)
- **normalized** — Unicode-нормализация и удаление HTML-тегов
- **preprocessed** — полная предобработка (колонка `text_preprocessed`)

Фильтрация по кириллице: остаются только сообщения, где доля кириллических символов превышает 30%.

In [4]:
variants = prepare_text_variants(df)

ru_variants = {}
for name, vdf in variants.items():
    mask = vdf['text'].apply(lambda t: is_mostly_cyrillic(str(t)))
    ru_variants[name] = vdf[mask].reset_index(drop=True)
    print(f'{name}: {len(ru_variants[name])} сообщений')

raw: 72342 сообщений
normalized: 72355 сообщений
preprocessed: 72503 сообщений


## Дообучение RuBERT-tiny2

Модель: `cointegrated/rubert-tiny2` — компактный BERT для русского языка.
Количество классов: 2 (ham=0, spam=1).

### Загрузка модели и токенайзера

In [5]:
MODEL_NAME = 'cointegrated/rubert-tiny2'
model_config = get_model_config(MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
rb_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

### Подготовка датасетов и токенизация

Для каждого варианта текста создаётся отдельный train/test сплит (80/20) и токенизируется.

In [6]:
tokenized = {}
for name, vdf in ru_variants.items():
    ds = Dataset.from_pandas(vdf)
    split = ds.train_test_split(test_size=0.2, seed=42)
    train_ds = split['train']
    test_ds = split['test']

    train_tok = train_ds.map(
        tokenize_function,
        batched=True,
        fn_kwargs={'tokenizer': tokenizer, 'max_length': model_config['max_length']},
    )
    test_tok = test_ds.map(
        tokenize_function,
        batched=True,
        fn_kwargs={'tokenizer': tokenizer, 'max_length': model_config['max_length']},
    )

    tokenized[name] = {'train': train_tok, 'test': test_tok}
    print(f'{name}: train={len(train_tok)}, test={len(test_tok)}')

Map:   0%|          | 0/57873 [00:00<?, ? examples/s]

Map:   0%|          | 0/14469 [00:00<?, ? examples/s]

raw: train=57873, test=14469


Map:   0%|          | 0/57884 [00:00<?, ? examples/s]

Map:   0%|          | 0/14471 [00:00<?, ? examples/s]

normalized: train=57884, test=14471


Map:   0%|          | 0/58002 [00:00<?, ? examples/s]

Map:   0%|          | 0/14501 [00:00<?, ? examples/s]

preprocessed: train=58002, test=14501


### Создание trainer'ов

In [7]:
rubert_cfg = {
    'learning_rate': 2e-5,
    'epochs': 7,
    'batch_size': 16,
    'fp16': True,
    'focal_alpha': 0.25,
    'focal_gamma': 2.0,
}

trainers = {}
for name in tokenized:
    output_dir = f'rb_tiny2_{name}'

    training_args = get_training_args(
        output_dir=output_dir,
        learning_rate=rubert_cfg['learning_rate'],
        num_train_epochs=rubert_cfg['epochs'],
        per_device_train_batch_size=rubert_cfg['batch_size'],
        per_device_eval_batch_size=rubert_cfg['batch_size'],
        max_length=model_config['max_length'],
        fp16=rubert_cfg['fp16'],
        gradient_checkpointing=model_config['gradient_checkpointing'],
    )

    trainer = FocalLossTrainer(
        focal_alpha=rubert_cfg['focal_alpha'],
        focal_gamma=rubert_cfg['focal_gamma'],
        model=rb_model,
        args=training_args,
        train_dataset=tokenized[name]['train'],
        eval_dataset=tokenized[name]['test'],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainers[name] = trainer

### Запуск дообучения

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')
rb_model.to(device)

Устройство: cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), 

#### На исходных текстах (raw)

In [9]:
trainers['raw'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.001390,0.994056,0.984939,0.991887,0.978087
2,No log,0.001135,0.994609,0.986421,0.987452,0.985391
3,No log,0.001388,0.995231,0.987948,0.992281,0.983652
4,No log,0.001583,0.994955,0.987218,0.994006,0.980522
5,No log,0.001722,0.995300,0.988137,0.991250,0.985043
6,No log,0.001861,0.995439,0.988458,0.994020,0.982957
7,No log,0.001851,0.995577,0.988811,0.994025,0.983652


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=25326, training_loss=0.001491129101805426, metrics={'train_runtime': 549.602, 'train_samples_per_second': 737.099, 'train_steps_per_second': 46.081, 'total_flos': 1493686181127168.0, 'train_loss': 0.001491129101805426, 'epoch': 7.0})

In [10]:
trainers['raw'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.001851,7,0.995577,0.988811,0.994025,0.983652


{'eval_loss': 0.0018511671805754304,
 'eval_accuracy': 0.9955767502937314,
 'eval_f1': 0.9888111888111888,
 'eval_precision': 0.99402460456942,
 'eval_recall': 0.9836521739130435}

Сохранение модели.

In [11]:
rb_model.save_pretrained(MODELS_DIR / 'finetuned_rubert_tiny2_raw')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rubert_tiny2_raw')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rubert_tiny2_raw/tokenizer_config.json',
 '/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rubert_tiny2_raw/tokenizer.json')

#### На нормализованных текстах (normalized)

In [12]:
trainers['normalized'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.000918,0.995923,0.989485,0.988956,0.990014
2,No log,0.000857,0.997236,0.992883,0.990767,0.995007
3,No log,0.001119,0.995854,0.989347,0.985149,0.993581
4,No log,0.001301,0.996268,0.990374,0.990021,0.990728


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=14472, training_loss=0.00038813378285017377, metrics={'train_runtime': 312.3664, 'train_samples_per_second': 1297.156, 'train_steps_per_second': 81.078, 'total_flos': 853697193197568.0, 'train_loss': 0.00038813378285017377, 'epoch': 4.0})

In [13]:
trainers['normalized'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.000857,4,0.997236,0.992883,0.990767,0.995007


{'eval_loss': 0.0008574804523959756,
 'eval_accuracy': 0.9972358510123696,
 'eval_f1': 0.9928825622775801,
 'eval_precision': 0.9907670454545454,
 'eval_recall': 0.9950071326676176}

Сохранение модели.

In [14]:
rb_model.save_pretrained(MODELS_DIR / 'finetuned_rubert_tiny2_norm')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rubert_tiny2_norm')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rubert_tiny2_norm/tokenizer_config.json',
 '/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rubert_tiny2_norm/tokenizer.json')

#### На предобработанных текстах (preprocessed)

In [15]:
trainers['preprocessed'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.000859,0.996207,0.990291,0.993976,0.986634
2,No log,0.000881,0.996552,0.991231,0.988458,0.994020
3,No log,0.000940,0.997104,0.992590,0.995752,0.989448
4,No log,0.001180,0.997104,0.992595,0.995051,0.990151
5,No log,0.001225,0.996897,0.992073,0.993649,0.990503
6,No log,0.001352,0.996897,0.992065,0.994696,0.989448


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=21756, training_loss=0.00039060143430260616, metrics={'train_runtime': 470.7847, 'train_samples_per_second': 862.42, 'train_steps_per_second': 53.914, 'total_flos': 1283156259065856.0, 'train_loss': 0.00039060143430260616, 'epoch': 6.0})

In [16]:
trainers['preprocessed'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.001180,6,0.997104,0.992595,0.995051,0.990151


{'eval_loss': 0.001180117018520832,
 'eval_accuracy': 0.9971036480242742,
 'eval_f1': 0.9925952045133991,
 'eval_precision': 0.9950512548603747,
 'eval_recall': 0.9901512486809708}

Сохранение модели.

In [17]:
rb_model.save_pretrained(MODELS_DIR / 'finetuned_rubert_tiny2_p')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rubert_tiny2_p')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rubert_tiny2_p/tokenizer_config.json',
 '/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rubert_tiny2_p/tokenizer.json')

## Оценка качества моделей

Оценка выполняется на наборе тестовых сообщений с помощью Hugging Face `pipeline`.
Для каждого варианта текста выводятся предсказания.

Тестовые сообщения

In [18]:
test_messages = [
    "Это честное сообщение от пользователя.",
    "🔥 Казино онлайн! Зарабатывай миллионы прямо сейчас! 💰💎",
    "Зарабатывай миллионы **онлайн** прямо сейчас!",
    "Работа на дому, легкий доход. Пиши в личку!",
    "Привет! Как дела? У меня всё отлично.",
    "Discover the hidden secrets of the digital market that top traders don't want you to know! I'm seeking five motivated individuals who are committed to earning over $100K weekly in the digital market. Once you start seeing profits, I'll require just 15% of your earnings as my fee. Please note: I'm only interested in working with five serious and dedicated people should send me a direct message or ask me (HOW) via TELEGRAM\n\nhttps://t.me/ancleroyofficial",
    "steam gift 50$ - steamcommunity.com/gift-card/pay/50\n@everyone",
    "Давайте **вместе** будем писать про казино в чатах!!! Присоединяйтесь!",
    "Как же надоели эти сообщения про казино",
    "Добрый день. Для подачи документов необходимо пройти регистрацию здесь: stankin.ru",
    "Добрый день. Для подачи документов необходимо пройти регистрацию здесь: https://stankin.ru",
    "Поступление — это почти что казино! Лотерея!",
    "3-4 часа и 8 тысяч твои!  Пиши  https://t.me/rasmuswork1",
    "Выиграл 345к дало x3450\n\nИграл тут: @jet_casino_ibot",
]

In [19]:
classifiers = {}
suffix_map = {'raw': 'raw', 'normalized': 'norm', 'preprocessed': 'p'}
for name, suffix in suffix_map.items():
    model_path = str(MODELS_DIR / f'finetuned_rubert_tiny2_{suffix}')
    classifiers[name] = pipeline('text-classification', model=model_path, tokenizer=model_path)

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

In [20]:
for name, classifier in classifiers.items():
    print(f'--- {name} ---')
    for msg in test_messages:
        result = classifier(msg)
        print(f'{result}  |  {msg[:80]}')
    print()

--- raw ---
[{'label': 'LABEL_0', 'score': 0.9771261811256409}]  |  Это честное сообщение от пользователя.
[{'label': 'LABEL_1', 'score': 0.966877281665802}]  |  🔥 Казино онлайн! Зарабатывай миллионы прямо сейчас! 💰💎
[{'label': 'LABEL_1', 'score': 0.7528031468391418}]  |  Зарабатывай миллионы **онлайн** прямо сейчас!
[{'label': 'LABEL_1', 'score': 0.989150881767273}]  |  Работа на дому, легкий доход. Пиши в личку!
[{'label': 'LABEL_0', 'score': 0.9532455205917358}]  |  Привет! Как дела? У меня всё отлично.
[{'label': 'LABEL_1', 'score': 0.919277548789978}]  |  Discover the hidden secrets of the digital market that top traders don't want yo
[{'label': 'LABEL_1', 'score': 0.9105992913246155}]  |  steam gift 50$ - steamcommunity.com/gift-card/pay/50
@everyone
[{'label': 'LABEL_0', 'score': 0.7090471386909485}]  |  Давайте **вместе** будем писать про казино в чатах!!! Присоединяйтесь!
[{'label': 'LABEL_0', 'score': 0.7721208333969116}]  |  Как же надоели эти сообщения про казино
[{'label':

## Замер CPU-инференса

Замеряется latency инференса на CPU для оценки пригодности модели к развёртыванию на сервере без GPU.

In [21]:
sample_texts = test_messages * 10
cpu_results = {}
for name, suffix in suffix_map.items():
    model_path = str(MODELS_DIR / f'finetuned_rubert_tiny2_{suffix}')
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    tok = AutoTokenizer.from_pretrained(model_path)
    cpu_results[name] = benchmark_cpu_inference(model, tok, sample_texts, max_length=model_config['max_length'])
    print(f'{name}: {cpu_results[name]}')

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

raw: {'avg_ms_per_msg': 2.04, 'p95_ms_per_msg': 4.15, 'throughput_msgs_per_sec': 489.45}


Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

normalized: {'avg_ms_per_msg': 1.79, 'p95_ms_per_msg': 1.85, 'throughput_msgs_per_sec': 557.58}


Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

preprocessed: {'avg_ms_per_msg': 1.85, 'p95_ms_per_msg': 1.93, 'throughput_msgs_per_sec': 540.31}
